# Phase 2 · Notebook 03 — Fine-Tuning RoBERTa on TAB

This is the contender. Phase 1 and the previous two Phase 2 notebooks all use models trained on *general-purpose* English text. None of them have ever seen a TAB-style label set, so they are doomed to fail on `CODE`, `DEM`, and the long tail of MISC categories.

Here we fine-tune `roberta-base` on TAB's training split (~1,014 documents, ~50,000 entity mentions). The hypothesis is straightforward: a model trained on the exact label set we care about will dominate every off-the-shelf baseline on those labels.

**Compute notes**
- T4 (Colab free) — ~30 minutes for 3 epochs
- M-series Mac (MPS) — ~60–90 minutes
- CPU — 4–6 hours; not recommended

The notebook detects what's available and adapts.

---


In [1]:
# ── Run me first if you're on Colab (skip locally — already in requirements.txt) ──
# !pip -q install transformers datasets evaluate seqeval accelerate \
#                 presidio-analyzer presidio-anonymizer scikit-learn spacy
# !python -m spacy download en_core_web_lg
# !python -m spacy download en_core_web_sm


## Setup


In [2]:
import sys
sys.path.insert(0, "../src")

import os
import time
import numpy as np
import pandas as pd
import torch
import warnings
warnings.filterwarnings("ignore")

from anonymisation.data import load_tab
from anonymisation.iob import (
    BIO_LABELS, LABEL_TO_ID, ID_TO_LABEL, TAB_ENTITY_TYPES,
    gold_spans_for_training, offsets_to_bio,
)
from anonymisation.evaluation import (
    evaluate_document, merge_results, results_to_dataframe,
)
from anonymisation.predictors import make_finetuned_predictor
from anonymisation.device import best_device, report_device

print(report_device())
print(f"BIO label set ({len(BIO_LABELS)} labels):", BIO_LABELS[:5], "...")


Device: mps    (Apple Silicon (MPS))
BIO label set (17 labels): ['O', 'B-CODE', 'I-CODE', 'B-DATETIME', 'I-DATETIME'] ...


## Configuration


In [3]:
BASE_MODEL  = "roberta-base"
MAX_LENGTH  = 384       # RoBERTa positional limit is 512; leave headroom
STRIDE      = 64        # overlap between sliding windows over long docs
NUM_EPOCHS  = 3
LR          = 2e-5
BATCH_SIZE  = 16        # T4-friendly; reduce to 8 on MPS / CPU
SEED        = 42

OUTPUT_DIR  = "../models/roberta-tab"
RESULTS_PATH = "../results/finetune_roberta.csv"

# Reduce-per-step config based on detected device
device, _ = best_device()
if device != "cuda":
    BATCH_SIZE = 8
    print(f"Non-CUDA device — reducing batch size to {BATCH_SIZE}")


Non-CUDA device — reducing batch size to 8


## Load TAB


In [4]:
dataset = load_tab()
print({split: len(dataset[split]) for split in dataset})


{'train': 1112, 'validation': 541, 'test': 555}


## Tokenise & build BIO-tagged training examples

Each TAB document is potentially long (tens of thousands of characters). We sliding-window each doc into 384-token chunks with 64 tokens of overlap, then convert character-offset entity_mentions into per-token BIO label IDs using `anonymisation.iob.offsets_to_bio`.

Subword continuations are labelled `-100` so they don't contribute to the cross-entropy loss — the standard recipe.


In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, add_prefix_space=True)


def tokenise_and_align(doc):
    text = doc["text"]
    spans = gold_spans_for_training(doc)

    enc = tokenizer(
        text,
        return_offsets_mapping=True,
        return_overflowing_tokens=True,
        truncation=True,
        max_length=MAX_LENGTH,
        stride=STRIDE,
        padding=False,
    )

    examples = []
    for chunk_idx in range(len(enc["input_ids"])):
        offsets = enc["offset_mapping"][chunk_idx]
        word_ids = enc.word_ids(batch_index=chunk_idx)
        labels = offsets_to_bio(offsets, spans, word_ids=word_ids)

        examples.append({
            "input_ids":      enc["input_ids"][chunk_idx],
            "attention_mask": enc["attention_mask"][chunk_idx],
            "labels":         labels,
        })
    return examples


def build_split(split_name, max_docs=None):
    out = []
    docs = list(dataset[split_name])
    if max_docs:
        docs = docs[:max_docs]
    for doc in docs:
        out.extend(tokenise_and_align(doc))
    return out


print("Tokenising train + validation (this takes ~1 min)...")
train_examples = build_split("train")
val_examples   = build_split("validation")
print(f"  train chunks: {len(train_examples):,}")
print(f"  val   chunks: {len(val_examples):,}")


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenising train + validation (this takes ~1 min)...
  train chunks: 6,372
  val   chunks: 1,913


In [6]:
# Wrap as a HuggingFace Dataset for the Trainer
from datasets import Dataset

train_ds = Dataset.from_list(train_examples)
val_ds   = Dataset.from_list(val_examples)


## Build the model


In [7]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(BIO_LABELS),
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
)
print(f"Model: {BASE_MODEL}, num_labels={len(BIO_LABELS)}")


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: roberta-base, num_labels=17


## Train


In [8]:
from transformers import (
    DataCollatorForTokenClassification, TrainingArguments, Trainer
)
import evaluate as hf_evaluate

collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
seqeval = hf_evaluate.load("seqeval")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    true_labels, true_preds = [], []
    for p_seq, l_seq in zip(preds, labels):
        tl, tp = [], []
        for p, l in zip(p_seq, l_seq):
            if l == -100:
                continue
            tl.append(ID_TO_LABEL[int(l)])
            tp.append(ID_TO_LABEL[int(p)])
        true_labels.append(tl)
        true_preds.append(tp)
    res = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "f1":        res["overall_f1"],
        "precision": res["overall_precision"],
        "recall":    res["overall_recall"],
    }


args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    seed=SEED,
    report_to="none",
    fp16=(device == "cuda"),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()
print("\nDone. Best checkpoint loaded.")


Starting training...


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.096900,0.101920,0.788714,0.823059,0.757120
2,0.069100,0.103559,0.788875,0.798330,0.779642
3,0.068100,0.103641,0.809963,0.819724,0.800431



Done. Best checkpoint loaded.


## Evaluate on TAB test (same framework as Phase 1)

We reuse the character-offset evaluation framework from Phase 1 — that way the per-entity-type numbers are directly comparable to spaCy / HF / Presidio.


In [9]:
predict = make_finetuned_predictor(model, tokenizer, device=device, max_length=MAX_LENGTH, stride=STRIDE)

test_docs = list(dataset["test"])
print(f"Evaluating on {len(test_docs)} test documents...")

all_merged = {}
for mode in ["partial", "exact"]:
    print(f"\n--- {mode} match ---")
    per_doc = []
    start = time.time()
    for i, doc in enumerate(test_docs):
        if (i + 1) % 50 == 0:
            elapsed = time.time() - start
            print(f"  {i + 1}/{len(test_docs)}   ({(i + 1)/elapsed:.1f} docs/s)")
        per_doc.append(evaluate_document(predict, doc, mode=mode))
    print(f"  done in {time.time() - start:.1f}s")
    all_merged[mode] = merge_results(per_doc)

results_df = results_to_dataframe(all_merged)
results_df.insert(0, "model", "roberta_finetuned_tab")
results_df.to_csv(RESULTS_PATH, index=False)
print(f"\nSaved → {RESULTS_PATH}")


Evaluating on 555 test documents...

--- partial match ---
  50/555   (6.3 docs/s)
  100/555   (6.8 docs/s)
  150/555   (7.0 docs/s)
  200/555   (7.4 docs/s)
  250/555   (7.9 docs/s)
  300/555   (8.5 docs/s)
  350/555   (8.9 docs/s)
  400/555   (9.5 docs/s)
  450/555   (10.1 docs/s)
  500/555   (9.6 docs/s)
  550/555   (9.4 docs/s)
  done in 59.2s

--- exact match ---
  50/555   (8.4 docs/s)
  100/555   (8.2 docs/s)
  150/555   (8.0 docs/s)
  200/555   (8.2 docs/s)
  250/555   (8.7 docs/s)
  300/555   (9.3 docs/s)
  350/555   (9.7 docs/s)
  400/555   (10.3 docs/s)
  450/555   (10.9 docs/s)
  500/555   (10.3 docs/s)
  550/555   (10.0 docs/s)
  done in 55.7s

Saved → results/finetuned_results.csv


## Per-entity results


In [10]:
from anonymisation.mapping import TAB_TO_SPACY

for mode in ["partial", "exact"]:
    merged = all_merged[mode]
    print(f"\n── {mode.upper()} MATCH ──")
    rows = []
    for et in list(TAB_TO_SPACY.keys()) + ["_ALL"]:
        r = merged[et]
        rows.append({
            "Entity": et if et != "_ALL" else "▶ OVERALL",
            "TP": r.tp, "FP": r.fp, "FN": r.fn,
            "Precision": f"{r.precision:.1%}",
            "Recall":    f"{r.recall:.1%}",
            "F1":        f"{r.f1:.1%}",
        })
    print(pd.DataFrame(rows).to_string(index=False))



── PARTIAL MATCH ──
   Entity    TP   FP   FN Precision Recall    F1
   PERSON  3841  670  297     85.1%  92.8% 88.8%
      ORG  1238  639  708     66.0%  63.6% 64.8%
      LOC  1159  345  300     77.1%  79.4% 78.2%
 DATETIME  9250  824  286     91.8%  97.0% 94.3%
 QUANTITY   515  258  149     66.6%  77.6% 71.7%
     CODE  1587  242   25     86.8%  98.4% 92.2%
      DEM   351  278  566     55.8%  38.3% 45.4%
     MISC    45  218  492     17.1%   8.4% 11.2%
▶ OVERALL 17986 3474 2823     83.8%  86.4% 85.1%

── EXACT MATCH ──
   Entity    TP   FP   FN Precision Recall    F1
   PERSON  3546  965  592     78.6%  85.7% 82.0%
      ORG  1022  855  924     54.4%  52.5% 53.5%
      LOC  1063  441  396     70.7%  72.9% 71.8%
 DATETIME  8886 1188  650     88.2%  93.2% 90.6%
 QUANTITY   361  412  303     46.7%  54.4% 50.2%
     CODE  1435  394  177     78.5%  89.0% 83.4%
      DEM   244  385  673     38.8%  26.6% 31.6%
     MISC    17  246  520      6.5%   3.2%  4.3%
▶ OVERALL 16574 4886 4235    

## Save the model

Optional — only do this if you'll need the trained weights later (the head-to-head notebook only reads the results CSV, not the model itself).


In [12]:
trainer.save_model(OUTPUT_DIR + "/final")
tokenizer.save_pretrained(OUTPUT_DIR + "/final")
print(f"Saved → {OUTPUT_DIR}/final")


Saved → checkpoints/roberta-tab/final


## What to look for

- **CODE recall should jump dramatically** — from spaCy's 0% to something high (this is the most direct test of the fine-tune-vs-off-the-shelf hypothesis).
- **DEM and MISC should improve substantially** — TAB labels these specifically; the model has now seen them.
- **PERSON should be ≥ spaCy** but probably not by a huge margin — both models have plenty of training data for that label.
- **QUANTITY precision should improve** — the fine-tune learns that *most* numbers in legal text are NO_MASK and shouldn't be predicted at all. This is where the false-positive flood from Phase 1 should subside.

If the overall F1 lands above 0.80 partial-match, that's a credible "production-ready with caveats" headline for the writeup.

If it lands lower than that, the next things I'd try:
- More epochs (5 or 7) — TAB train is small; a second pass over each doc usually helps.
- A bigger backbone (`roberta-large`) — same recipe, more parameters.
- A document-level CRF head on top — helps with span boundary errors.

Move on to `04_head_to_head.ipynb` to compare against the baselines.
